In [ ]:
import requests
import pandas as pd
import time


def get_cryptocompare_daily(symbol, days=1500):
    """
    CryptoCompare Free API - No API key required
    Maximum 2000 days of historical data
    """
    url = "https://min-api.cryptocompare.com/data/v2/histoday"
    params = {
        "fsym": symbol,
        "tsym": "USD",
        "limit": min(days, 2000),
    }
    resp = requests.get(url, params=params)
    data = resp.json()
   
    if data.get("Response") != "Success":
        print(f" Error for {symbol}: {data.get('Message')}")
        return None
   
    records = data["Data"]["Data"]
    df = pd.DataFrame(records)
    df["date"] = pd.to_datetime(df["time"], unit="s").dt.date
    df = df.set_index("date")["close"]
    df = df[df > 0]  # Filter out early zero values
    return df


# Asset mapping closer to LlamaRisk's original report
ASSETS = {
    "ETH": "ETH",   # Corresponds to WETH
    "BTC": "BTC",   # Corresponds to WBTC / cbBTC
    "LINK": "LINK", # Available on Aave
    "AAVE": "AAVE", # Available on Aave
    "MKR": "MKR",   # Available on Aave
    "UNI": "UNI",   # Available on Aave
    "CRV": "CRV",   # Available on Aave, with rich risk history
}


all_prices = {}
for name, symbol in ASSETS.items():
    print(f"Fetching {name}...")
    series = get_cryptocompare_daily(symbol, days=1500)
    if series is not None:
        all_prices[name] = series
        print(f" {name}: {len(series)} days, "
              f"from {series.index[0]} to {series.index[-1]}")
    time.sleep(0.3)


price_df = pd.DataFrame(all_prices).dropna()
print(f"\nPrice Matrix Dimensions: {price_df.shape}")
price_df.to_csv("crypto_prices_llamarisk.csv")

In [ ]:
# Test these two symbols
for symbol in ["WSTETH", "WEETH", "RETH", "MKR", "UNI", "CRV"]:
    series = get_cryptocompare_daily(symbol, days=1500)
    if series is not None:
        print(f"{symbol}: {len(series)} days")

In [ ]:
import pandas as pd
import numpy as np
from hmmlearn import hmm
import time
import os


# ====================== File Paths ======================
CSV_FILE = "crypto_prices_llamarisk.csv"


# ====================== Load Data or Download ======================
if os.path.exists(CSV_FILE):
    print(f"✅ Loading data from local CSV: {CSV_FILE}")
    price_df = pd.read_csv(CSV_FILE, index_col='date', parse_dates=True)
    print(f"Price matrix shape: {price_df.shape}")
else:
    print("⚠️ CSV file not found, downloading data from API...")
    # ←←← Paste your previous full data download code here (Binance or yfinance version) ←←←
   
    # For example, using the Binance version you had before:
   
    # ... (paste the Binance download function and loop here) ...
   
    price_df = pd.DataFrame(all_prices).dropna(how='all')
    price_df.to_csv(CSV_FILE)
    print(f"✅ Data saved to {CSV_FILE}")


# ====================== HMM Regime Detection ======================
print("\nStarting HMM Market Stress Regime Detection...")

In [ ]:
from hmmlearn import hmm
import numpy as np
import pandas as pd


# Log daily returns
returns = np.log(price_df / price_df.shift(1)).dropna()
assets = list(ASSETS.keys())


# HMM Regime Identification
vol_feature = returns.std(axis=1).values.reshape(-1, 1)

model = hmm.GaussianHMM(n_components=2, covariance_type="full",
                         n_iter=1000, random_state=42)
model.fit(vol_feature)
hidden_states = model.predict(vol_feature)

# High volatility = Stress Regime
state_vol = {s: vol_feature[hidden_states == s].mean() for s in [0, 1]}
stress_state = max(state_vol, key=state_vol.get)
normal_state = 1 - stress_state

regime_label = pd.Series(hidden_states, index=returns.index).map(
    {stress_state: "stress", normal_state: "normal"}
)

ret_stress = returns[regime_label == "stress"]
ret_normal = returns[regime_label == "normal"]


print(f"Stress regime days: {(regime_label == 'stress').sum()} "
      f"({(regime_label == 'stress').mean() * 100:.1f}%)")
print(f"Normal regime days: {(regime_label == 'normal').sum()} "
      f"({(regime_label == 'normal').mean() * 100:.1f}%)")
print(f"\nAverage cross-sectional volatility (Stress Regime): "
      f"{vol_feature[hidden_states == stress_state].mean():.5f}")
print(f"Average cross-sectional volatility (Normal Regime): "
      f"{vol_feature[hidden_states == normal_state].mean():.5f}")

In [ ]:
from scipy import interpolate


def compute_stieltjes(eigenvalues, z_grid):
    return np.array([np.mean(1.0 / (z - eigenvalues)) for z in z_grid], dtype=complex)


def cov_to_corr(cov):
    std = np.sqrt(np.diag(cov))
    return cov / np.outer(std, std)


def free_additive_deconvolution(Sigma_observed, Sigma_normal, n_grid=400):
    N = Sigma_observed.shape[0]
    
    evals_obs = np.linalg.eigvalsh(Sigma_observed)
    evals_normal = np.linalg.eigvalsh(Sigma_normal)
    
    # Normalize eigenvalues
    evals_obs = evals_obs / evals_obs.mean()
    evals_normal = evals_normal / evals_normal.mean()
    
    y_grid = np.linspace(0.05, 5.0, n_grid)
    z_grid = 1j * y_grid
    
    G_obs = compute_stieltjes(evals_obs, z_grid)
    G_normal = compute_stieltjes(evals_normal, z_grid)
    
    R_obs = z_grid - 1.0 / G_obs
    R_normal = z_grid - 1.0 / G_normal
    R_stress = R_obs - R_normal
    
    # R_stress → Spectral density (fixed-point iteration)
    w_mag = np.abs(G_obs)
    sort_idx = np.argsort(w_mag)
    
    R_real_fn = interpolate.interp1d(w_mag[sort_idx],
                                      np.real(R_stress)[sort_idx],
                                      bounds_error=False, fill_value=0.0)
    R_imag_fn = interpolate.interp1d(w_mag[sort_idx],
                                      np.imag(R_stress)[sort_idx],
                                      bounds_error=False, fill_value=0.0)
    
    def R_func(w):
        mag = np.abs(w)
        return R_real_fn(mag) + 1j * R_imag_fn(mag)
    
    epsilon = 0.04
    x_grid = np.linspace(1e-3, 6.0, 600)
    z_target = x_grid + 1j * epsilon
    G_st = 1.0 / z_target
    
    for _ in range(200):
        G_new = 1.0 / (z_target - R_func(G_st))
        if np.max(np.abs(G_new - G_st)) < 1e-9:
            break
        G_st = G_new
    
    rho_stress = np.maximum(-np.imag(G_st) / np.pi, 0)
    norm = np.trapz(rho_stress, x_grid)
    if norm > 0:
        rho_stress /= norm
    
    print(f"Spectral density integral (should be 1 after normalization): "
          f"{np.trapz(rho_stress, x_grid):.4f}")
    
    # Discretize into N eigenvalues
    cdf = np.cumsum(rho_stress)
    cdf /= cdf[-1]
    quantiles = np.linspace(1/(2*N), 1 - 1/(2*N), N)
    stress_evals = np.interp(quantiles, cdf, x_grid)
    stress_evals = np.maximum(stress_evals, 1e-6)
    
    # Restore original scale
    scale = np.trace(Sigma_observed) / N
    stress_evals_scaled = stress_evals * scale
    
    _, evecs_obs = np.linalg.eigh(Sigma_observed)
    Sigma_stress = evecs_obs @ np.diag(stress_evals_scaled) @ evecs_obs.T
    
    return {
        "Sigma_stress": Sigma_stress,
        "stress_evals": stress_evals_scaled,
        "x_grid": x_grid,
        "rho_stress": rho_stress,
        "y_grid": y_grid,
        "R_obs": R_obs,
        "R_normal": R_normal,
        "R_stress": R_stress,
    }


# ── Execution ───────────────────────────────────────────────
Sigma_observed = returns[assets].cov().values
Sigma_normal = ret_normal[assets].cov().values

print("Running Free Probability Deconvolution...")
fp = free_additive_deconvolution(Sigma_observed, Sigma_normal)
Sigma_stress_FP = fp["Sigma_stress"]


# Validation Output
std_obs = np.sqrt(np.diag(Sigma_observed))
std_normal = np.sqrt(np.diag(Sigma_normal))
std_stress = np.sqrt(np.diag(Sigma_stress_FP))

print(f"\n{'Asset':<8} {'σ_mixed':>10} {'σ_normal':>10} {'σ_stress(FP)':>14}")
for i, a in enumerate(assets):
    print(f"{a:<8} {std_obs[i]:>10.5f} {std_normal[i]:>10.5f} {std_stress[i]:>14.5f}")


# Average Correlation Comparison
mask = ~np.eye(len(assets), dtype=bool)
print(f"\nAverage Correlation Comparison:")
print(f"  Mixed Regime (Current LlamaRisk): {cov_to_corr(Sigma_observed)[mask].mean():.4f}")
print(f"  Normal Regime: {cov_to_corr(Sigma_normal)[mask].mean():.4f}")
print(f"  Stress Regime (FP Extracted): {cov_to_corr(Sigma_stress_FP)[mask].mean():.4f}")

In [ ]:
def free_additive_deconvolution_v2(Sigma_observed, Sigma_normal, n_grid=400):
    N = Sigma_observed.shape[0]
    evals_obs = np.linalg.eigvalsh(Sigma_observed)
    evals_normal = np.linalg.eigvalsh(Sigma_normal)
    
    # ── Record original scale for final restoration ──
    scale_obs = evals_obs.mean()
    scale_normal = evals_normal.mean()
    
    evals_obs_n = evals_obs / scale_obs
    evals_normal_n = evals_normal / scale_normal
    
    y_grid = np.linspace(0.05, 5.0, n_grid)
    z_grid = 1j * y_grid
    
    G_obs = compute_stieltjes(evals_obs_n, z_grid)
    G_normal = compute_stieltjes(evals_normal_n, z_grid)
    
    R_obs = z_grid - 1.0 / G_obs
    R_normal = z_grid - 1.0 / G_normal
    
    # ── Key Fix: Align R-transform to the same coordinate system ──
    # Σ_obs = Σ_normal ⊞ Σ_stress
    # R-transform is linear, but the two matrices were normalized with different scales.
    # We need to rescale R_normal back to the observed coordinate system before subtraction.
    scale_ratio = scale_normal / scale_obs
    R_normal_aligned = R_normal * scale_ratio
    R_stress = R_obs - R_normal_aligned
    
    # ── Diagnostic Output ──
    print(f" scale_obs: {scale_obs:.6f}")
    print(f" scale_normal: {scale_normal:.6f}")
    print(f" scale_ratio: {scale_ratio:.4f}")
    print(f" R_obs mean (real part): {np.real(R_obs).mean():.4f}")
    print(f" R_normal mean (real part): {np.real(R_normal_aligned).mean():.4f}")
    print(f" R_stress mean (real part): {np.real(R_stress).mean():.4f}")
    
    # Fixed-point iteration to reconstruct G_stress
    w_mag = np.abs(G_obs)
    sort_idx = np.argsort(w_mag)
    
    R_real_fn = interpolate.interp1d(w_mag[sort_idx],
                                      np.real(R_stress)[sort_idx],
                                      bounds_error=False,
                                      fill_value=(np.real(R_stress)[sort_idx][0],
                                                  np.real(R_stress)[sort_idx][-1]))
    R_imag_fn = interpolate.interp1d(w_mag[sort_idx],
                                      np.imag(R_stress)[sort_idx],
                                      bounds_error=False,
                                      fill_value=(np.imag(R_stress)[sort_idx][0],
                                                  np.imag(R_stress)[sort_idx][-1]))
    
    def R_func(w):
        mag = np.abs(w)
        return R_real_fn(mag) + 1j * R_imag_fn(mag)
    
    epsilon = 0.04
    x_grid = np.linspace(1e-3, 8.0, 800)  # Expanded range to avoid truncation
    z_target = x_grid + 1j * epsilon
    G_st = 1.0 / z_target
    
    for iteration in range(300):
        G_new = 1.0 / (z_target - R_func(G_st))
        err = np.max(np.abs(G_new - G_st))
        if err < 1e-9:
            print(f" Fixed-point iteration converged at step {iteration}")
            break
        G_st = G_new
    
    rho_stress = np.maximum(-np.imag(G_st) / np.pi, 0)
    norm = np.trapz(rho_stress, x_grid)
    print(f" Spectral density integral before normalization: {norm:.4f}")
    
    if norm > 0:
        rho_stress /= norm
    
    # ── Discretization: Generate N eigenvalues ──
    cdf = np.cumsum(rho_stress)
    cdf /= cdf[-1]
    quantiles = np.linspace(1/(2*N), 1 - 1/(2*N), N)
    stress_evals_normalized = np.interp(quantiles, cdf, x_grid)
    stress_evals_normalized = np.maximum(stress_evals_normalized, 1e-6)
    
    # ── Scale Restoration: Stress should be larger than observed ──
    # Σ_stress trace ≈ Σ_obs trace - Σ_normal trace (under free addition)
    trace_stress_expected = np.trace(Sigma_observed) - np.trace(Sigma_normal)
    print(f"\n Trace Decomposition Diagnostics:")
    print(f" tr(Σ_obs): {np.trace(Sigma_observed):.6f}")
    print(f" tr(Σ_normal): {np.trace(Sigma_normal):.6f}")
    print(f" Expected tr(Σ_stress): {trace_stress_expected:.6f}")
    
    if trace_stress_expected <= 0:
        # Fallback: use direct sample covariance from stress regime
        print(" ⚠️ Negative trace difference, falling back to direct sample estimation")
        Sigma_stress_direct = ret_stress[assets].cov().values
        return _build_result(Sigma_stress_direct, fp_used=False,
                             x_grid=x_grid, rho_stress=rho_stress,
                             y_grid=y_grid, R_obs=R_obs,
                             R_normal=R_normal_aligned, R_stress=R_stress)
    
    scale_stress = trace_stress_expected / N
    stress_evals_scaled = stress_evals_normalized * scale_stress
    
    _, evecs_obs = np.linalg.eigh(Sigma_observed)
    Sigma_stress = evecs_obs @ np.diag(stress_evals_scaled) @ evecs_obs.T
    
    return {
        "Sigma_stress": Sigma_stress,
        "stress_evals": stress_evals_scaled,
        "x_grid": x_grid,
        "rho_stress": rho_stress,
        "y_grid": y_grid,
        "R_obs": R_obs,
        "R_normal": R_normal_aligned,
        "R_stress": R_stress,
        "fp_used": True,
    }


# ── Execution ───────────────────────────────────────────────
print("Running Free Probability Deconvolution v2...")
fp = free_additive_deconvolution_v2(Sigma_observed, Sigma_normal)
Sigma_stress_FP = fp["Sigma_stress"]

std_obs = np.sqrt(np.diag(Sigma_observed))
std_normal = np.sqrt(np.diag(Sigma_normal))
std_stress = np.sqrt(np.diag(Sigma_stress_FP))

print(f"\n{'Asset':<8} {'σ_mixed':>10} {'σ_normal':>10} {'σ_stress(FP)':>14}")
for i, a in enumerate(assets):
    print(f"{a:<8} {std_obs[i]:>10.5f} {std_normal[i]:>10.5f} {std_stress[i]:>14.5f}")

mask = ~np.eye(len(assets), dtype=bool)
print(f"\nAverage Correlation Comparison:")
print(f"  Mixed Regime (Current LlamaRisk): {cov_to_corr(Sigma_observed)[mask].mean():.4f}")
print(f"  Normal Regime: {cov_to_corr(Sigma_normal)[mask].mean():.4f}")
print(f"  Stress Regime (FP Extracted): {cov_to_corr(Sigma_stress_FP)[mask].mean():.4f}")

In [ ]:
# ── Correct Framework: Weighted Mixture instead of Free Additive Convolution ──

p_stress = (regime_label == "stress").mean()   # e.g. 0.156
p_normal = (regime_label == "normal").mean()   # e.g. 0.844

# Validate the mixture relationship
Sigma_reconstructed = p_normal * Sigma_normal + p_stress * ret_stress[assets].cov().values
Sigma_stress_direct = ret_stress[assets].cov().values

print("Mixture Relationship Validation:")
print(f" tr(Σ_obs): {np.trace(Sigma_observed):.6f}")
print(f" tr(Reconstructed): {np.trace(Sigma_reconstructed):.6f}")
print(f" tr(Σ_stress_direct): {np.trace(Sigma_stress_direct):.6f}")


# ── What RMT Can Actually Contribute: Noise Cleaning ──
def marchenko_pastur_clean(cov_matrix, T):
    """
    Clean the covariance matrix using Marchenko-Pastur law
    Remove noise eigenvalues and retain true signal
    """
    N = cov_matrix.shape[0]
    std = np.sqrt(np.diag(cov_matrix))
    corr = cov_matrix / np.outer(std, std)
    
    eigenvalues, eigenvectors = np.linalg.eigh(corr)
    eigenvalues = eigenvalues[::-1]
    eigenvectors = eigenvectors[:, ::-1]
    
    q = N / T
    lambda_max_mp = (1 + np.sqrt(q)) ** 2   # MP upper bound
    
    signal_mask = eigenvalues > lambda_max_mp
    noise_mean = eigenvalues[~signal_mask].mean()
    
    cleaned = eigenvalues.copy()
    cleaned[~signal_mask] = noise_mean
    
    corr_clean = eigenvectors @ np.diag(cleaned) @ eigenvectors.T
    d = np.sqrt(np.diag(corr_clean))
    corr_clean /= np.outer(d, d)
    
    cov_clean = corr_clean * np.outer(std, std)
    n_signal = signal_mask.sum()
    
    print(f" Samples T={T}, Assets N={N}, q={q:.3f}")
    print(f" MP Threshold: {lambda_max_mp:.4f}")
    print(f" Signal eigenvalues: {n_signal}, Noise eigenvalues: {N - n_signal}")
    print(f" Eigenvalues: {np.round(eigenvalues, 3)}")
    
    return cov_clean, corr_clean, eigenvalues, signal_mask, lambda_max_mp


# Clean all three matrices
print("\n=== Mixed Regime (Current LlamaRisk) ===")
Sigma_obs_clean, Corr_obs_clean, evals_obs, mask_obs, mp_obs = \
    marchenko_pastur_clean(Sigma_observed, len(returns))

print("\n=== Normal Regime ===")
Sigma_normal_clean, Corr_normal_clean, evals_normal, mask_normal, mp_normal = \
    marchenko_pastur_clean(Sigma_normal, len(ret_normal))

print("\n=== Stress Regime (Direct Sample) ===")
Sigma_stress_direct = ret_stress[assets].cov().values
Sigma_stress_clean, Corr_stress_clean, evals_stress, mask_stress, mp_stress = \
    marchenko_pastur_clean(Sigma_stress_direct, len(ret_stress))


# ── Final Comparison ───────────────────────────────────────
mask = ~np.eye(len(assets), dtype=bool)

print(f"\n{'':30} {'Avg Correlation':>12} {'Max Eigenvalue':>14}")
for label, corr in [
    ("Mixed Regime - Raw (Current LlamaRisk)", cov_to_corr(Sigma_observed)),
    ("Mixed Regime - RMT Cleaned", Corr_obs_clean),
    ("Stress Regime - Direct Sample", cov_to_corr(Sigma_stress_direct)),
    ("Stress Regime - RMT Cleaned", Corr_stress_clean),
]:
    evals = np.linalg.eigvalsh(corr)
    print(f" {label:30} {corr[mask].mean():>12.4f} {evals.max():>14.4f}")


print(f"\n{'Asset':<8}", end="")
for label in ["σ_mixed", "σ_normal", "σ_stress"]:
    print(f"{label:>12}", end="")
print()

for i, a in enumerate(assets):
    print(f"{a:<8}", end="")
    for cov in [Sigma_observed, Sigma_normal_clean, Sigma_stress_clean]:
        print(f"{np.sqrt(cov[i,i]):>12.5f}", end="")
    print()

In [ ]:
# ── Quantifying the Dual Bias in LlamaRisk's Current Method ──

def generate_shock_scenarios(corr_matrix, vol_vector,
                             n_scenarios=10000, var99_mult=2.33):
    """
    Generate shock scenarios using Cholesky decomposition
    """
    N = corr_matrix.shape[0]
    eigvals, eigvecs = np.linalg.eigh(corr_matrix)
    eigvals = np.maximum(eigvals, 1e-8)
    corr_psd = eigvecs @ np.diag(eigvals) @ eigvecs.T
    
    # Ensure positive semi-definite
    d = np.sqrt(np.diag(corr_psd))
    corr_psd /= np.outer(d, d)
    
    L = np.linalg.cholesky(corr_psd)
    Z = np.random.randn(n_scenarios, N)
    return (Z @ L.T) * vol_vector * var99_mult


np.random.seed(42)


# LlamaRisk Current Method: Mixed Correlation + Mixed Volatility
scenarios_current = generate_shock_scenarios(
    Corr_obs_clean,
    np.sqrt(np.diag(Sigma_observed))
)

# RMT Improved Method: Stress Correlation + Stress Volatility
scenarios_improved = generate_shock_scenarios(
    Corr_stress_clean,
    np.sqrt(np.diag(Sigma_stress_clean))
)


# Asset-level VaR99 Analysis
print(f"\n{'Asset':<8} {'VaR99_Current':>12} {'VaR99_Improved':>12} "
      f"{'Vol Ratio':>12} {'Driver':>12}")
print("-" * 60)

for i, a in enumerate(assets):
    v_cur = np.percentile(scenarios_current[:, i], 1) * 100
    v_imp = np.percentile(scenarios_improved[:, i], 1) * 100
   
    # Volatility-only effect
    vol_ratio = np.sqrt(Sigma_stress_clean[i,i]) / np.sqrt(Sigma_observed[i,i])
   
    print(f"{a:<8} {v_cur:>11.2f}% {v_imp:>11.2f}% "
          f"{vol_ratio:>11.2f}x {'↑Volatility Dominant':>12}")


# Portfolio-level Analysis
print("\n=== Portfolio Level: Average Loss in Worst 1% Scenarios ===")
portfolio_current = scenarios_current.mean(axis=1)
portfolio_improved = scenarios_improved.mean(axis=1)

cvar_current = np.percentile(portfolio_current, 1)
cvar_improved = np.percentile(portfolio_improved, 1)

print(f" Current Method CVaR99 (Equal-Weighted): {cvar_current*100:.2f}%")
print(f" RMT Improved CVaR99 (Equal-Weighted): {cvar_improved*100:.2f}%")
print(f" Difference: {(cvar_improved - cvar_current)*100:.2f}pp")
print(f" Improved method is more conservative: {cvar_improved < cvar_current}")


# Joint Tail Risk Analysis
print("\n=== Joint Tail Loss Probability (Two Assets Drop >5% Simultaneously) ===")
print(f"{'Asset Pair':<15} {'Current':>10} {'RMT Improved':>10} {'Diff':>8}")

threshold = -0.05
pairs = [("ETH","BTC"), ("ETH","LINK"), ("BTC","MKR"), ("CRV","AAVE")]
asset_idx = {a: i for i, a in enumerate(assets)}

for a1, a2 in pairs:
    i1, i2 = asset_idx[a1], asset_idx[a2]
    p_cur = ((scenarios_current[:,i1] < threshold) &
             (scenarios_current[:,i2] < threshold)).mean()
    p_imp = ((scenarios_improved[:,i1] < threshold) &
             (scenarios_improved[:,i2] < threshold)).mean()
    print(f"{a1+'-'+a2:<15} {p_cur:>9.3f}% {p_imp:>9.3f}% {p_imp-p_cur:>+7.3f}%")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
fig = plt.figure(figsize=(18, 12))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# ── Figure 1: Regime Identification ──
ax1 = fig.add_subplot(gs[0, :2])
vol_ts = returns.std(axis=1)
idx = list(range(len(vol_ts)))
ax1.plot(idx, vol_ts.values, color="steelblue", lw=0.8, alpha=0.8)
stress_mask = (regime_label == "stress").values
ax1.fill_between(idx, vol_ts.values, 0,
                 where=stress_mask,
                 color="red", alpha=0.35, label="Stress Regime")
ax1.fill_between(idx, vol_ts.values, 0,
                 where=~stress_mask,
                 color="green", alpha=0.15, label="Normal Regime")
ax1.set_title("HMM Regime Identification (2022–2026)\n"
              "Stress regime: 15.6% of days, avg volatility 2.25x normal",
              fontsize=11)
ax1.set_ylabel("Cross-sectional Volatility")
ax1.set_xlabel("Trading Day")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.2)

# ── Figure 2: Eigenvalue Spectrum ──
ax2 = fig.add_subplot(gs[0, 2])
x_pos = np.arange(len(assets))
width = 0.28
ax2.bar(x_pos - width, evals_obs,    width, label="Mixed Regime",  color="gray",  alpha=0.8)
ax2.bar(x_pos,         evals_normal, width, label="Normal Regime", color="green", alpha=0.8)
ax2.bar(x_pos + width, evals_stress, width, label="Stress Regime", color="red",   alpha=0.8)
ax2.axhline(mp_stress, color="red",   linestyle="--", lw=1.2,
            label=f"MP bound (stress)={mp_stress:.2f}")
ax2.axhline(mp_obs,    color="black", linestyle=":",  lw=1.0,
            label=f"MP bound (mixed)={mp_obs:.2f}")
ax2.set_xticks(x_pos)
ax2.set_xticklabels([f"λ{i+1}" for i in range(len(assets))], fontsize=8)
ax2.set_title("Eigenvalue Spectrum by Regime\n(normalized correlation matrix)", fontsize=11)
ax2.set_ylabel("Eigenvalue")
ax2.legend(fontsize=7)
ax2.grid(True, alpha=0.2)

# ── Figure 3 & 4: Correlation Matrices ──
for col, (cov, title) in enumerate([
    (Sigma_observed,      "Mixed-Regime Correlation\n(LlamaRisk Current)"),
    (Sigma_stress_direct, "Stress-Regime Correlation\n(RMT-Separated)"),
]):
    ax = fig.add_subplot(gs[1, col])
    corr = cov_to_corr(cov)
    im = ax.imshow(corr, cmap="RdYlGn", vmin=0.3, vmax=1.0)
    ax.set_xticks(range(len(assets)))
    ax.set_yticks(range(len(assets)))
    ax.set_xticklabels(assets, rotation=45, fontsize=8)
    ax.set_yticklabels(assets, fontsize=8)
    ax.set_title(title, fontsize=11)
    plt.colorbar(im, ax=ax, fraction=0.046)
    for i in range(len(assets)):
        for j in range(len(assets)):
            ax.text(j, i, f"{corr[i,j]:.2f}",
                    ha="center", va="center", fontsize=6,
                    color="black" if corr[i,j] < 0.85 else "white")

# ── Figure 5: VaR99 Comparison ──
ax4 = fig.add_subplot(gs[1, 2])
var_current  = [np.percentile(scenarios_current[:,i],  1)*100 for i in range(len(assets))]
var_improved = [np.percentile(scenarios_improved[:,i], 1)*100 for i in range(len(assets))]

x = np.arange(len(assets))
ax4.bar(x - 0.2, var_current,  0.38, label="Current Method", color="steelblue", alpha=0.85)
ax4.bar(x + 0.2, var_improved, 0.38, label="RMT-Improved",   color="red",       alpha=0.85)

for i in range(len(assets)):
    diff_pct = (var_improved[i] - var_current[i]) / abs(var_current[i]) * 100
    ax4.text(i, min(var_current[i], var_improved[i]) - 1.5,
             f"{diff_pct:+.0f}%", ha="center", fontsize=7, color="darkred")

ax4.set_xticks(x)
ax4.set_xticklabels(assets, fontsize=9)
ax4.set_ylabel("VaR99 (%)")
gap=np.percentile(scenarios_improved.mean(axis=1),1)*100- np.percentile(scenarios_current.mean(axis=1),1)*100
ax4.set_title(f"VaR99 Comparison\n"
              f"Portfolio CVaR: {np.percentile(scenarios_current.mean(axis=1),1)*100:.1f}% "
              f"→ {np.percentile(scenarios_improved.mean(axis=1),1)*100:.1f}% "
              f"(gap: {gap:.2f}pp)", fontsize=10)
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.2, axis="y")
ax4.axhline(0, color="black", lw=0.5)

plt.suptitle(
    "RMT Regime Separation: Improving LlamaRisk's Debt Ceiling Methodology\n"
    "Stress-regime volatility systematically exceeds mixed estimates (1.4x–1.7x), "
    "causing 40%–70% underestimation of VaR99",
    fontsize=13, y=1.01
)

plt.savefig("rmt_debt_ceiling_report——llama.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: rmt_debt_ceiling_report——llama.png")

In [ ]:
# ============================================================
# Sensitivity Analysis: How CVaR Difference Changes with Different Stress Regime Thresholds
# ============================================================

from hmmlearn import hmm
import numpy as np
import pandas as pd


def run_regime_analysis(returns, assets, n_stress_pct_target=None, random_state=42):
    """
    Re-run regime analysis with different stress definitions.
    Using volatility percentile thresholds to directly define stress regimes.
    """
    vol_feature = returns.std(axis=1)
    results = []
   
    # Test different volatility percentile thresholds to define stress regime
    for pct in [5, 10, 15, 20, 25, 30]:
        threshold = np.percentile(vol_feature, 100 - pct)
        stress_mask = vol_feature >= threshold
       
        ret_stress_s = returns[stress_mask]
        ret_normal_s = returns[~stress_mask]
       
        if len(ret_stress_s) < 20 or len(ret_normal_s) < 20:
            continue
       
        Sigma_obs_s = returns[assets].cov().values
        Sigma_stress_s = ret_stress_s[assets].cov().values
       
        # Marchenko-Pastur cleaning function
        def mp_clean(cov, T):
            N = cov.shape[0]
            std = np.sqrt(np.diag(cov))
            corr = cov / np.outer(std, std)
            evals, evecs = np.linalg.eigh(corr)
            evals = evals[::-1]
            evecs = evecs[:, ::-1]
            
            q = N / T
            lam_max = (1 + np.sqrt(q)) ** 2
            noise_mean = evals[evals <= lam_max].mean()
            
            cleaned = np.where(evals > lam_max, evals, noise_mean)
            corr_c = evecs @ np.diag(cleaned) @ evecs.T
            d = np.sqrt(np.diag(corr_c))
            corr_c /= np.outer(d, d)
            return corr_c * np.outer(std, std), corr_c
       
        _, Corr_obs_c = mp_clean(Sigma_obs_s, len(returns))
        _, Corr_stress_c = mp_clean(Sigma_stress_s, len(ret_stress_s))
       
        vol_obs = np.sqrt(np.diag(Sigma_obs_s))
        vol_stress = np.sqrt(np.diag(Sigma_stress_s))
       
        np.random.seed(42)
        sc_cur = generate_shock_scenarios(Corr_obs_c, vol_obs, 10000)
        sc_imp = generate_shock_scenarios(Corr_stress_c, vol_stress, 10000)
       
        cvar_cur = np.percentile(sc_cur.mean(axis=1), 1) * 100
        cvar_imp = np.percentile(sc_imp.mean(axis=1), 1) * 100
        gap = cvar_imp - cvar_cur
       
        results.append({
            "stress_pct": pct,
            "n_stress_days": stress_mask.sum(),
            "vol_threshold": threshold,
            "cvar_current": cvar_cur,
            "cvar_improved": cvar_imp,
            "gap_pp": gap,
            "vol_ratio": vol_stress.mean() / vol_obs.mean(),
        })
       
        print(f" Stress Threshold = {pct:2d}% percentile | "
              f"Days = {stress_mask.sum():4d} | "
              f"CVaR Current = {cvar_cur:+.2f}% | "
              f"CVaR Improved = {cvar_imp:+.2f}% | "
              f"Gap = {gap:+.2f}pp")
   
    return pd.DataFrame(results)


print("=== Sensitivity Analysis: CVaR Gap under Different Stress Regime Definitions ===\n")

sensitivity_df = run_regime_analysis(returns, assets)

print(f"\n{'Stress %':>8} {'Stress Days':>10} {'CVaR Current':>12} "
      f"{'CVaR Improved':>12} {'Gap (pp)':>10} {'Vol Ratio':>10}")
print("-" * 70)

for _, row in sensitivity_df.iterrows():
    print(f"{row['stress_pct']:>7.0f}% "
          f"{row['n_stress_days']:>10.0f} "
          f"{row['cvar_current']:>11.2f}% "
          f"{row['cvar_improved']:>11.2f}% "
          f"{row['gap_pp']:>+9.2f}pp "
          f"{row['vol_ratio']:>9.2f}x")

print(f"\nConclusion:")
print(f" Gap Range: {sensitivity_df['gap_pp'].min():.2f}pp ~ "
      f"{sensitivity_df['gap_pp'].max():.2f}pp")
print(f" Conclusion is robust: {'Yes' if sensitivity_df['gap_pp'].max() < 0 else 'No, direction inconsistent'}")

In [ ]:
# ============================================================
# Historical Backtest: Validate Whether RMT-Improved VaR99 Covers Actual Losses
# Logic: Use past N days to estimate VaR99, then check if the actual loss on day N+1 falls within the range
# ============================================================

def rolling_var99(returns_df, assets, window=252, n_scenarios=5000):
    """
    Rolling VaR99 estimation
    For each trading day: use the past 'window' days of data
    to estimate next day's VaR99 (Current method vs RMT Improved)
    """
    results = []
   
    for t in range(window, len(returns_df) - 1):
        # Historical window
        hist = returns_df.iloc[t-window:t]
        next_day = returns_df.iloc[t+1]        # Actual next-day returns
        
        # Regime identification (using volatility percentile within historical window)
        vol_hist = hist.std(axis=1)
        stress_threshold = np.percentile(vol_hist, 85)   # Top 15% as stress
        stress_mask = vol_hist >= stress_threshold
       
        ret_stress_h = hist[stress_mask]
        ret_normal_h = hist[~stress_mask]
       
        if len(ret_stress_h) < 15:
            continue
       
        Sigma_obs_h = hist[assets].cov().values
        Sigma_stress_h = ret_stress_h[assets].cov().values
       
        # Simplified Marchenko-Pastur cleaning
        def quick_clean(cov, T):
            N = cov.shape[0]
            std = np.sqrt(np.diag(cov))
            corr = cov / np.outer(std, std)
            evals, evecs = np.linalg.eigh(corr)
            evals = evals[::-1]
            evecs = evecs[:, ::-1]
            
            lam_max = (1 + np.sqrt(N / T)) ** 2
            noise_mean = evals[evals <= lam_max].mean() if (evals <= lam_max).any() else evals.min()
            
            cleaned = np.where(evals > lam_max, evals, noise_mean)
            corr_c = evecs @ np.diag(cleaned) @ evecs.T
            d = np.sqrt(np.diag(corr_c))
            corr_c /= np.outer(d, d)
            return corr_c * np.outer(std, std), corr_c
       
        _, Corr_obs_h = quick_clean(Sigma_obs_h, len(hist))
        _, Corr_stress_h = quick_clean(Sigma_stress_h, len(ret_stress_h))
       
        vol_obs_h = np.sqrt(np.diag(Sigma_obs_h))
        vol_stress_h = np.sqrt(np.diag(Sigma_stress_h))
       
        np.random.seed(t)
        sc_cur = generate_shock_scenarios(Corr_obs_h, vol_obs_h, n_scenarios)
        sc_imp = generate_shock_scenarios(Corr_stress_h, vol_stress_h, n_scenarios)
       
        # Equal-weighted portfolio VaR99
        var99_cur = np.percentile(sc_cur.mean(axis=1), 1)
        var99_imp = np.percentile(sc_imp.mean(axis=1), 1)
       
        # Actual next-day equal-weighted portfolio return
        actual_ret = next_day[assets].mean()
       
        results.append({
            "date": returns_df.index[t+1],
            "var99_cur": var99_cur,
            "var99_imp": var99_imp,
            "actual_ret": actual_ret,
            "breach_cur": actual_ret < var99_cur,      # Actual loss exceeds VaR99
            "breach_imp": actual_ret < var99_imp,
        })
   
    return pd.DataFrame(results)


print("Running rolling backtest (may take 1-2 minutes)...")
backtest_df = rolling_var99(returns, assets, window=252, n_scenarios=3000)


# ── Backtest Statistics ─────────────────────────────────────
n_obs = len(backtest_df)
breach_cur = backtest_df["breach_cur"].sum()
breach_imp = backtest_df["breach_imp"].sum()
breach_rate_cur = breach_cur / n_obs * 100
breach_rate_imp = breach_imp / n_obs * 100

print(f"\n=== Historical Backtest Results ===")
print(f"Backtest days: {n_obs}")
print(f"Theoretical VaR99 breach rate: 1.00% (max 1 breach per 100 days)")
print(f"\nCurrent Method:")
print(f"  Breaches: {breach_cur}")
print(f"  Breach rate: {breach_rate_cur:.2f}%")
print(f"  Assessment: {'✓ Pass' if breach_rate_cur <= 1.5 else '✗ Fail (Underestimating risk)'}")
print(f"\nRMT Improved Method:")
print(f"  Breaches: {breach_imp}")
print(f"  Breach rate: {breach_rate_imp:.2f}%")
print(f"  Assessment: {'✓ Pass' if breach_rate_imp <= 1.5 else '✗ Fail (Underestimating risk)'}")


# ── List Breach Dates (Stress Events) ───────────────────────
print(f"\n=== Current Method Breach Dates (Actual loss exceeds VaR99) ===")
breaches = backtest_df[backtest_df["breach_cur"]]
for _, row in breaches.iterrows():
    print(f" {row['date'].date()}  Actual={row['actual_ret']*100:+.2f}%  "
          f"VaR99_Current={row['var99_cur']*100:.2f}%  "
          f"RMT_Coverage={'✓' if row['actual_ret'] >= row['var99_imp'] else '✗'}")

In [ ]:
# ============================================================
# Historical Backtest: Validate Whether RMT-Improved VaR99 Covers Actual Losses
# Using Historical Simulation Method (No Monte Carlo Scaling)
# ============================================================

def rolling_var99_historical(returns_df, assets, window=252):
    """
    Historical Simulation VaR99 (No Monte Carlo)
    Directly uses the distribution of past 'window' days returns to estimate VaR99
    Comparison: Mixed Regime vs Pure Stress Regime
    """
    results = []
   
    for t in range(window, len(returns_df) - 1):
        hist = returns_df.iloc[t-window:t]
        next_day = returns_df.iloc[t+1]
       
        # Equal-weighted portfolio historical returns
        port_hist = hist[assets].mean(axis=1)
       
        # Regime identification
        vol_hist = hist.std(axis=1)
        stress_threshold = np.percentile(vol_hist, 85)
        stress_mask = vol_hist >= stress_threshold
       
        port_stress = port_hist[stress_mask]
       
        if len(port_stress) < 10:
            continue
       
        # Method A: Current LlamaRisk Method (VaR99 from full historical data)
        var99_cur = np.percentile(port_hist, 1)
       
        # Method B: RMT Improved (VaR99 from stress regime only)
        # Since stress samples are limited, use mean + std with normal assumption (2.33σ)
        mu_stress = port_stress.mean()
        std_stress = port_stress.std()
        var99_imp = mu_stress - 2.33 * std_stress
       
        actual_ret = next_day[assets].mean()
       
        results.append({
            "date": returns_df.index[t+1],
            "var99_cur": var99_cur,
            "var99_imp": var99_imp,
            "actual_ret": actual_ret,
            "breach_cur": actual_ret < var99_cur,
            "breach_imp": actual_ret < var99_imp,
            "is_stress": vol_hist.iloc[-1] >= stress_threshold,
        })
   
    return pd.DataFrame(results)


print("Running Historical Simulation Backtest...")

bt = rolling_var99_historical(returns, assets, window=252)

n_obs = len(bt)
breach_cur = bt["breach_cur"].sum()
breach_imp = bt["breach_imp"].sum()
br_rate_cur = breach_cur / n_obs * 100
br_rate_imp = breach_imp / n_obs * 100

print(f"\n=== Historical Simulation Backtest Results ===")
print(f"Backtest days: {n_obs}")
print(f"Theoretical VaR99 breach rate: 1.00%")
print(f"\n{'Method':<20} {'Breaches':>8} {'Breach Rate':>10} {'Assessment':>12}")
print("-" * 55)
print(f"{'Current Method':<20} {breach_cur:>8} {br_rate_cur:>9.2f}% "
      f"{'✓ Pass' if br_rate_cur <= 1.5 else '✗ Underestimating Risk'}")
print(f"{'RMT Improved':<20} {breach_imp:>8} {br_rate_imp:>9.2f}% "
      f"{'✓ Pass' if br_rate_imp <= 1.5 else '✗ Underestimating Risk'}")


# Breach distribution by regime
bt_stress = bt[bt["is_stress"]]
bt_normal = bt[~bt["is_stress"]]

print(f"\n=== Breach Breakdown by Regime ===")
print(f"Stress Days Breach (Current Method): "
      f"{bt_stress['breach_cur'].sum()}/{len(bt_stress)} = "
      f"{bt_stress['breach_cur'].mean()*100:.2f}%")
print(f"Normal Days Breach (Current Method): "
      f"{bt_normal['breach_cur'].sum()}/{len(bt_normal)} = "
      f"{bt_normal['breach_cur'].mean()*100:.2f}%")
print(f"\nStress Days Breach (RMT Improved): "
      f"{bt_stress['breach_imp'].sum()}/{len(bt_stress)} = "
      f"{bt_stress['breach_imp'].mean()*100:.2f}%")
print(f"Normal Days Breach (RMT Improved): "
      f"{bt_normal['breach_imp'].sum()}/{len(bt_normal)} = "
      f"{bt_normal['breach_imp'].mean()*100:.2f}%")


# List breach dates
print(f"\n=== Current Method Breach Dates (Actual loss > VaR99) ===")
breaches = bt[bt["breach_cur"]].head(20)
for _, row in breaches.iterrows():
    rmt_covered = "✓ RMT Covered" if not row["breach_imp"] else "✗ RMT Also Failed"
    print(f" {row['date'].date()}  Actual={row['actual_ret']*100:+.2f}%  "
          f"VaR99_Current={row['var99_cur']*100:.2f}%  "
          f"VaR99_Improved={row['var99_imp']*100:.2f}%  {rmt_covered}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
import pandas as pd
import numpy as np

plt.style.use("default")

fig = plt.figure(figsize=(18, 14))
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)


# ── Figure 1: Sensitivity Analysis ──
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(sensitivity_df["stress_pct"],
         sensitivity_df["gap_pp"].abs(),
         "o-", color="#F44336", lw=2, markersize=8, label="|CVaR Gap|")
ax1.fill_between(sensitivity_df["stress_pct"],
                 sensitivity_df["gap_pp"].abs(),
                 alpha=0.15, color="#F44336")
ax1.axhline(12.96, color="steelblue", linestyle="--", lw=1.5,
            label="HMM baseline (12.96pp)")

for _, row in sensitivity_df.iterrows():
    ax1.annotate(f"{row['gap_pp']:.1f}pp",
                 (row["stress_pct"], abs(row["gap_pp"])),
                 textcoords="offset points", xytext=(0, 8),
                 ha="center", fontsize=8)

ax1.set_xlabel("Stress Regime Threshold (top X% volatility)")
ax1.set_ylabel("CVaR Gap (percentage points)")
ax1.set_title("Sensitivity Analysis\nCVaR Gap vs Stress Threshold Definition",
              fontsize=10)
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.2)


# ── Figure 2: Volatility Ratio vs CVaR Gap ──
ax2 = fig.add_subplot(gs[0, 1])
ax2.scatter(sensitivity_df["vol_ratio"],
            sensitivity_df["gap_pp"].abs(),
            c=sensitivity_df["stress_pct"],
            cmap="RdYlGn_r", s=120, zorder=5)

for _, row in sensitivity_df.iterrows():
    ax2.annotate(f"{row['stress_pct']:.0f}%",
                 (row["vol_ratio"], abs(row["gap_pp"])),
                 textcoords="offset points", xytext=(5, 3), fontsize=8)

ax2.set_xlabel("Stress/Normal Volatility Ratio")
ax2.set_ylabel("CVaR Gap (pp)")
ax2.set_title("Volatility Ratio vs CVaR Gap\n(color = stress threshold %)",
              fontsize=10)
ax2.grid(True, alpha=0.2)


# ── Figure 3: Breach Rate Comparison ──
ax3 = fig.add_subplot(gs[0, 2])
categories = ["Overall", "Stress Days", "Normal Days"]
rates_cur = [br_rate_cur,
             bt["breach_cur"][bt["is_stress"]].mean()*100,
             bt["breach_cur"][~bt["is_stress"]].mean()*100]
rates_imp = [br_rate_imp,
             bt["breach_imp"][bt["is_stress"]].mean()*100,
             bt["breach_imp"][~bt["is_stress"]].mean()*100]

x = np.arange(len(categories))
ax3.bar(x - 0.2, rates_cur, 0.38, label="Current Method",
        color="steelblue", alpha=0.85)
ax3.bar(x + 0.2, rates_imp, 0.38, label="RMT-Improved",
        color="#F44336", alpha=0.85)

ax3.axhline(1.0, color="black", linestyle="--", lw=1.5,
            label="Theoretical 1% bound")

for i, (rc, ri) in enumerate(zip(rates_cur, rates_imp)):
    ax3.text(i - 0.2, rc + 0.05, f"{rc:.2f}%",
             ha="center", fontsize=8, color="steelblue")
    ax3.text(i + 0.2, ri + 0.05, f"{ri:.2f}%",
             ha="center", fontsize=8, color="#F44336")

ax3.set_xticks(x)
ax3.set_xticklabels(categories, fontsize=9)
ax3.set_ylabel("Breach Rate (%)")
ax3.set_title("VaR99 Breach Rate by Regime\n"
              "RMT halves overall breach rate (1.20% → 0.56%)", fontsize=10)
ax3.legend(fontsize=8)
ax3.grid(True, alpha=0.2, axis="y")


# ── Figure 4: Time Series Backtest (Spans Full Row) ──
ax4 = fig.add_subplot(gs[1, :])
bt_plot = bt.copy()
bt_plot["date"] = pd.to_datetime(bt_plot["date"])
bt_plot = bt_plot.set_index("date")

ax4.plot(bt_plot.index, bt_plot["actual_ret"]*100,
         color="gray", lw=0.7, alpha=0.6, label="Actual portfolio return")
ax4.plot(bt_plot.index, bt_plot["var99_cur"]*100,
         color="steelblue", lw=1.5, label="VaR99 Current Method")
ax4.plot(bt_plot.index, bt_plot["var99_imp"]*100,
         color="#F44336", lw=1.5, label="VaR99 RMT-Improved")

# Mark breach points
breaches_cur = bt_plot[bt_plot["breach_cur"]]
breaches_imp_only = bt_plot[bt_plot["breach_cur"] & ~bt_plot["breach_imp"]]
breaches_both = bt_plot[bt_plot["breach_cur"] & bt_plot["breach_imp"]]

ax4.scatter(breaches_imp_only.index,
            breaches_imp_only["actual_ret"]*100,
            color="green", s=80, zorder=5,
            label="Breach: Current only (RMT covers)")
ax4.scatter(breaches_both.index,
            breaches_both["actual_ret"]*100,
            color="black", s=80, marker="x", zorder=5,
            label="Breach: Both methods (black swan)")

# Mark major events
events = {
    "2024-04-13": "Iran-Israel\nGeopolitical",
    "2024-08-05": "Yen carry\nunwind",
    "2025-03-03": "Tariff war\nbegins",
    "2025-10-10": "Max drawdown\n-18.79%",
}

for date_str, label in events.items():
    try:
        date = pd.Timestamp(date_str)
        if date in bt_plot.index:
            y_val = bt_plot.loc[date, "actual_ret"]*100
            ax4.annotate(label, (date, y_val),
                        textcoords="offset points",
                        xytext=(0, -30), ha="center", fontsize=7,
                        arrowprops=dict(arrowstyle="->", color="black", lw=0.8))
    except:
        pass

ax4.axhline(0, color="black", lw=0.5)
ax4.set_ylabel("Return / VaR99 (%)")
ax4.set_title("Backtesting: Rolling VaR99 vs Actual Portfolio Returns (2023–2026)\n"
              "Green dots = breaches covered by RMT but missed by current method | "
              "Black × = black swan events missed by both",
              fontsize=10)
ax4.legend(fontsize=8, ncol=4)
ax4.grid(True, alpha=0.2)
ax4.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))


# ── Figure 5: Return Distribution by Regime ──
ax5 = fig.add_subplot(gs[2, :2])

stress_days_data = bt_plot[bt_plot["is_stress"]]
normal_days_data = bt_plot[~bt_plot["is_stress"]]

stress_actual = stress_days_data["actual_ret"].values * 100
normal_actual = normal_days_data["actual_ret"].values * 100

ax5.hist(normal_actual, bins=40, alpha=0.5, color="steelblue",
         density=True, label=f"Normal days (n={len(normal_actual)})")
ax5.hist(stress_actual, bins=20, alpha=0.6, color="#F44336",
         density=True, label=f"Stress days (n={len(stress_actual)})")

var99_normal_emp = np.percentile(normal_actual, 1)
var99_stress_emp = np.percentile(stress_actual, 1)

ax5.axvline(var99_normal_emp, color="steelblue", linestyle="--", lw=2,
            label=f"Normal VaR99 = {var99_normal_emp:.1f}%")
ax5.axvline(var99_stress_emp, color="#F44336", linestyle="--", lw=2,
            label=f"Stress VaR99 = {var99_stress_emp:.1f}%")

haircut = abs(var99_normal_emp) / abs(var99_stress_emp)

ax5.set_xlabel("Portfolio Return (%)")
ax5.set_ylabel("Density")
ax5.set_title(f"Return Distribution by Regime\n"
              f"Stress VaR99 ({var99_stress_emp:.1f}%) vs Normal VaR99 "
              f"({var99_normal_emp:.1f}%) → "
              f"Debt Ceiling Haircut = {haircut:.2f}x",
              fontsize=10)
ax5.legend(fontsize=8)
ax5.grid(True, alpha=0.2)


# ── Figure 6: Debt Ceiling Recommendation ──
ax6 = fig.add_subplot(gs[2, 2])

haircut = abs(var99_normal_emp) / abs(var99_stress_emp)
ceilings = [100, 100 * haircut, 100 * haircut * 0.85]

print(f"haircut={haircut:.3f}, ceilings={[round(c,1) for c in ceilings]}")

methods = ["Static\n(LlamaRisk)", "RMT Regime\nSeparation", "RMT + PT\nLiquidity"]
colors_bar = ["steelblue", "#FF9800", "#F44336"]

bars = ax6.bar(methods, ceilings, color=colors_bar, alpha=0.85, width=0.5)

for bar, val in zip(bars, ceilings):
    ax6.text(bar.get_x() + bar.get_width()/2,
             val + 1, f"${val:.0f}M\n({val-100:+.0f}%)",
             ha="center", fontsize=9, fontweight="bold")

ax6.set_ylabel("Debt Ceiling ($ Million, normalized to $100M baseline)")
ax6.set_title("Debt Ceiling Recommendation\nProgressive Conservatism", fontsize=10)
ax6.set_ylim(0, 115)
ax6.grid(True, alpha=0.2, axis="y")


# ── Overall Title ──
plt.suptitle(
    "RMT Improvements to LlamaRisk Debt Ceiling: Sensitivity, Backtest & Recommendation\n"
    "Robust across all stress thresholds (8.67–21.48pp gap) | "
    "Breach rate halved (1.20% → 0.56%) | "
    "Stress-day breach rate reduced (2.19% → 1.64%)",
    fontsize=11, y=1.01
)

plt.savefig("rmt_full_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

print("Saved: rmt_full_analysis.png")

In [ ]:
import requests
import json

# Explore Pendle Finance API - Check available market data
url = "https://api-v2.pendle.finance/core/v1/1/markets"

params = {
    "limit": 10,
    "is_expired": False
}

resp = requests.get(url, params=params)
data = resp.json()

# Print the full structure of the first market for reference
print("=== Pendle Market API Response Structure (First Market) ===")
print(json.dumps(data["results"][0], indent=2))

In [ ]:
import requests
import json

# Test historical data endpoints for Pendle Finance
market_addr = "0xc5b32dba5f29f8395fb9591e1a15f23a75214f33"

# Try several possible endpoints
endpoints = [
    f"https://api-v2.pendle.finance/core/v1/1/markets/{market_addr}/historical-data",
    f"https://api-v2.pendle.finance/core/v1/1/markets/{market_addr}/liquidity-history",
    f"https://api-v2.pendle.finance/core/v3/1/markets/{market_addr}/historical-data",
]

for url in endpoints:
    resp = requests.get(url)
    print(f"Status {resp.status_code}: {url}")
    
    if resp.status_code == 200:
        data = resp.json()
        # Print first 300 characters for preview
        preview = str(data)[:300] + "..." if len(str(data)) > 300 else str(data)
        print(json.dumps(preview, indent=2))
    else:
        print(f"  → No data or error")
    
    print("-" * 80)

In [ ]:
import requests
import json

market_addr = "0xc5b32dba5f29f8395fb9591e1a15f23a75214f33"

url = f"https://api-v2.pendle.finance/core/v3/1/markets/{market_addr}/historical-data"

resp = requests.get(url)
data = resp.json()

# Display summary information
print(f"Total records: {data['total']}")
print(f"Time range: {data['timestamp_start']} → {data['timestamp_end']}")

print(f"\nFirst record full fields:")
print(json.dumps(data['results'][0], indent=2))

print(f"\nLast record:")
print(json.dumps(data['results'][-1], indent=2))

In [ ]:
import requests
import pandas as pd
import time
import json

# Get all active Pendle markets
url = "https://api-v2.pendle.finance/core/v1/1/markets"
params = {
    "limit": 100,
    "is_expired": False,
    "order_by": "tvl:desc"
}

resp = requests.get(url, params=params)
markets = resp.json()["results"]

print(f"Total active markets: {len(markets)}")

print(f"\n{'Name':<30} {'TVL(M)':>10} {'Expiry Date':>12} {'Launch Date':>12}")
print("-" * 70)

target_markets = []

for m in markets:
    tvl = m.get("liquidity", {}).get("usd", 0)
    if tvl < 5_000_000:        # Filter: TVL < $5M
        continue
    
    name = m.get("proName", m.get("symbol", ""))[:28]
    expiry = m["expiry"][:10]
    created = m.get("timestamp", "")[:10]
    addr = m["address"]
   
    target_markets.append({
        "name": name,
        "address": addr,
        "expiry": expiry,
        "created": created,
        "tvl": tvl,
    })
    
    print(f"{name:<30} {tvl/1e6:>8.1f}M {expiry:>12} {created:>12}")

print(f"\nMarkets meeting criteria (TVL > $5M): {len(target_markets)}")

In [ ]:
resp = requests.get(url, params=params)
data = resp.json()
print(list(data.keys()))
print(json.dumps(str(data)[:500]))

In [ ]:
params = {"limit": 100, "is_expired": False}
resp = requests.get(url, params=params)
data = resp.json()
print(list(data.keys()))

In [ ]:
markets = data["results"]

print(f"Total active markets: {len(markets)}")

print(f"\n{'Name':<30} {'TVL(M)':>10} {'Expiry Date':>12} {'Launch Date':>12}")
print("-" * 70)

target_markets = []

for m in markets:
    tvl = m.get("liquidity", {}).get("usd", 0)
    if tvl < 5_000_000:          # Filter out markets with TVL < $5M
        continue
    
    name = m.get("proName", m.get("symbol", ""))[:28]
    expiry = m["expiry"][:10]
    created = m.get("timestamp", "")[:10]
    addr = m["address"]
   
    target_markets.append({
        "name": name,
        "address": addr,
        "expiry": expiry,
        "created": created,
        "tvl": tvl,
    })
    
    print(f"{name:<30} {tvl/1e6:>8.1f}M {expiry:>12} {created:>12}")

print(f"\nMarkets meeting criteria (TVL > $5M): {len(target_markets)}")

In [ ]:
def get_pt_historical(market_address):
    """
    Fetch historical data for a Pendle market
    """
    url = f"https://api-v2.pendle.finance/core/v3/1/markets/{market_address}/historical-data"
    resp = requests.get(url)
    
    if resp.status_code != 200:
        return None
    
    data = resp.json()
    if not data.get("results"):
        return None
   
    df = pd.DataFrame(data["results"])
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.set_index("timestamp")
    return df


print(f"{'Name':<30} {'Data Days':>10} {'Start Date':>12} "
      f"{'TVL Start(M)':>14} {'TVL Current(M)':>14}")
print("-" * 85)

market_data = {}

for m in target_markets:
    df = get_pt_historical(m["address"])
    time.sleep(0.3)
   
    if df is None:
        print(f"{m['name']:<30} {'No Data':>10}")
        continue
   
    # Resample to daily frequency (take last record of each day)
    df_daily = df.resample("D")["tvl"].last().dropna()
    n_days = len(df_daily)
   
    if n_days > 0:
        tvl_start = df_daily.iloc[0] / 1e6
        tvl_end = df_daily.iloc[-1] / 1e6
        date_start = df_daily.index[0].strftime("%Y-%m-%d")
        
        market_data[m["name"]] = df_daily
        
        print(f"{m['name']:<30} {n_days:>10} {date_start:>12} "
              f"{tvl_start:>13.1f}M {tvl_end:>13.1f}M")

print(f"\nSuccessfully fetched data for {len(market_data)} markets")

In [ ]:
import numpy as np


def cov_to_corr(cov):
    """Convert covariance matrix to correlation matrix"""
    std = np.sqrt(np.diag(cov))
    return cov / np.outer(std, std)


Corr_liq = cov_to_corr(Sigma_liq)

print("Liquidity Correlation Matrix:")
print(pd.DataFrame(Corr_liq, index=selected, columns=selected).round(3))

mask = ~np.eye(len(selected), dtype=bool)
print(f"\nAverage Liquidity Correlation: {Corr_liq[mask].mean():.4f}")

In [ ]:
# Filter for markets with 61 days of data
selected = ["USDG", "apxUSD", "apyUSD", "sNUSD", "reUSDe"]

liquidity_df = pd.DataFrame({
    name: market_data[name]
    for name in selected
    if name in market_data
}).dropna()

print(f"Liquidity matrix shape: {liquidity_df.shape}")
print(f"Time range: {liquidity_df.index[0]} → {liquidity_df.index[-1]}")

print(f"\nTVL Matrix ($M, first 5 rows):")
print((liquidity_df / 1e6).round(2).head())


# Calculate daily TVL returns (log returns)
liq_returns = np.log(liquidity_df / liquidity_df.shift(1)).dropna()

print(f"\nReturns matrix shape: {liq_returns.shape}")
print(f"N={liq_returns.shape[1]}, T={liq_returns.shape[0]}, "
      f"q=N/T={liq_returns.shape[1]/liq_returns.shape[0]:.3f}")


# Build liquidity covariance matrix
Sigma_liq = liq_returns.cov().values

print(f"\nLiquidity Covariance Matrix:")
print(pd.DataFrame(Sigma_liq,
                   index=selected,
                   columns=selected).round(4))


# Liquidity correlation matrix
Corr_liq = cov_to_corr(Sigma_liq)

print(f"\nLiquidity Correlation Matrix:")
print(pd.DataFrame(Corr_liq,
                   index=selected,
                   columns=selected).round(3))


# Average correlation
mask = ~np.eye(len(selected), dtype=bool)
print(f"\nAverage Liquidity Correlation: {Corr_liq[mask].mean():.4f}")

In [ ]:
# MP Law Noise Cleaning
print("=== RMT Analysis: PT Market Liquidity Matrix ===")

N = len(selected)
T = len(liq_returns)
q = N / T
lambda_max_mp = (1 + np.sqrt(q)) ** 2

print(f"N={N}, T={T}, q={q:.3f}")
print(f"MP Threshold: {lambda_max_mp:.4f}")

# Eigenvalue Decomposition
eigenvalues, eigenvectors = np.linalg.eigh(Corr_liq)
eigenvalues = eigenvalues[::-1]
eigenvectors = eigenvectors[:, ::-1]

print(f"\nEigenvalues:")
for i, ev in enumerate(eigenvalues):
    signal = "← Signal" if ev > lambda_max_mp else "← Noise"
    print(f" λ{i+1} = {ev:.4f} {signal}")

# Effective Rank
p = eigenvalues / eigenvalues.sum()
entropy = -np.sum(p * np.log(p + 1e-10))
eff_rank = np.exp(entropy)

print(f"\nEffective Rank: {eff_rank:.3f} / {N} ({eff_rank/N*100:.1f}%)")
print(f"Interpretation: Out of {N} PT markets, there are effectively {eff_rank:.1f} independent dimensions")


# Rolling Effective Rank (Capture Temporal Evolution)
window = 20
dates, eff_ranks, top_ratios = [], [], []

for i in range(window, len(liq_returns)):
    w_data = liq_returns.iloc[i-window:i]
    cov_w = w_data.cov().values
    corr_w = cov_to_corr(cov_w)
   
    ev_w = np.linalg.eigvalsh(corr_w)[::-1]
    ev_w = np.maximum(ev_w, 1e-8)
   
    p_w = ev_w / ev_w.sum()
    er_w = np.exp(-np.sum(p_w * np.log(p_w + 1e-10)))
    top_r = ev_w[0] / ev_w.sum()
   
    dates.append(liq_returns.index[i])
    eff_ranks.append(er_w)
    top_ratios.append(top_r)

rolling_df = pd.DataFrame({
    "effective_rank": eff_ranks,
    "top_eigenvalue_ratio": top_ratios
}, index=dates)

print(f"\nRolling Effective Rank (window={window} days):")
print(f"  Early period (first 10): {rolling_df['effective_rank'].iloc[:10].mean():.3f}")
print(f"  Recent period (last 10): {rolling_df['effective_rank'].iloc[-10:].mean():.3f}")
print(f"  Trend: {'Decreasing ↓ (Increasing Correlation)' if rolling_df['effective_rank'].iloc[-10:].mean() < rolling_df['effective_rank'].iloc[:10].mean() else 'Increasing ↑ (Decreasing Correlation)'}")

# Debt Ceiling Haircut Coefficient
systemic_early = rolling_df["top_eigenvalue_ratio"].iloc[:10].mean()
systemic_recent = rolling_df["top_eigenvalue_ratio"].iloc[-10:].mean()

haircut = max(0.5, min(1.0, 1 - (systemic_recent - systemic_early) * 3))

print(f"\n=== Debt Ceiling Recommendation ===")
print(f"Systemic Factor (Early): {systemic_early:.3f}")
print(f"Systemic Factor (Recent): {systemic_recent:.3f}")
print(f"Recommended Haircut Coefficient: {haircut:.3f}")
print(f"Interpretation: Static method suggests $100M → RMT method suggests ${haircut*100:.0f}M")

In [ ]:
# Quantify Liquidity Competition Effect
print("=== Liquidity Competition Structure Analysis ===\n")

# TVL Change per Market
tvl_change = (liquidity_df.iloc[-1] - liquidity_df.iloc[0]) / 1e6
tvl_pct = (liquidity_df.iloc[-1] / liquidity_df.iloc[0] - 1) * 100

print(f"{'Market':<10} {'TVL Start(M)':>14} {'TVL Current(M)':>14} "
      f"{'Change(M)':>12} {'Change%':>9}")
print("-" * 68)

for name in selected:
    start = liquidity_df[name].iloc[0] / 1e6
    end = liquidity_df[name].iloc[-1] / 1e6
    chg = end - start
    pct = (end / start - 1) * 100
    print(f"{name:<10} {start:>13.1f}M {end:>13.1f}M {chg:>+11.1f}M {pct:>+8.1f}%")


# LlamaRisk Current Method: 60-day Average vs Actual Current TVL
print(f"\n=== LlamaRisk Current Method (60-day Avg) vs Actual Current TVL ===")
print(f"{'Market':<10} {'60d Avg(M)':>14} {'Current TVL(M)':>14} {'Estimation Bias':>16}")
print("-" * 67)

for name in selected:
    avg_60 = liquidity_df[name].mean() / 1e6
    current = liquidity_df[name].iloc[-1] / 1e6
    overest = (avg_60 / current - 1) * 100
    direction = f"+{overest:.1f}% Overestimated" if overest > 0 else f"{overest:.1f}% Underestimated"
    print(f"{name:<10} {avg_60:>13.1f}M {current:>13.1f}M {direction:>20}")


# Competition Network: Who is competing for whose liquidity
print(f"\n=== Liquidity Competition Pairs (Negative Correlation) ===")
print(f"{'Market Pair':<28} {'Correlation':>10} {'Interpretation'}")
print("-" * 70)

for i in range(len(selected)):
    for j in range(i + 1, len(selected)):
        corr_val = Corr_liq[i, j]
        if corr_val < -0.1:
            m1_trend = "↑" if tvl_change[selected[i]] > 0 else "↓"
            m2_trend = "↑" if tvl_change[selected[j]] > 0 else "↓"
            print(f"{selected[i] + m1_trend + ' vs ' + selected[j] + m2_trend:<28} "
                  f"{corr_val:>10.3f}   Liquidity Competition")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

plt.style.use("default")  # Reset to white background

fig = plt.figure(figsize=(18, 11))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)


# ── Figure 1: TVL Time Series ──
ax1 = fig.add_subplot(gs[0, :2])
colors = ["#2196F3", "#FF9800", "#4CAF50", "#F44336", "#9C27B0", "#00BCD4"]

for i, name in enumerate(selected):
    ax1.plot(liquidity_df.index,
             liquidity_df[name] / 1e6,
             label=name, color=colors[i], lw=1.8)
    
    # 60-day average horizontal line
    avg = liquidity_df[name].mean() / 1e6
    ax1.axhline(avg, color=colors[i], lw=0.8, linestyle="--", alpha=0.5)

ax1.set_title("PT Market TVL Trajectories (2026-03-11 to 2026-05-10)\n"
              "Dashed lines = 60-day mean (LlamaRisk current method)",
              fontsize=11)
ax1.set_ylabel("TVL ($ Million)")
ax1.set_xlabel("Date")
ax1.legend(fontsize=8, ncol=3)
ax1.grid(True, alpha=0.2)


# ── Figure 2: Liquidity Correlation Matrix ──
ax2 = fig.add_subplot(gs[0, 2])
im = ax2.imshow(Corr_liq, cmap="RdYlGn", vmin=-0.5, vmax=1.0)

ax2.set_xticks(range(len(selected)))
ax2.set_yticks(range(len(selected)))
ax2.set_xticklabels(selected, rotation=45, fontsize=8)
ax2.set_yticklabels(selected, fontsize=8)
ax2.set_title("PT Market Liquidity\nCorrelation Matrix", fontsize=11)

plt.colorbar(im, ax=ax2, fraction=0.046)

# Annotate values
for i in range(len(selected)):
    for j in range(len(selected)):
        ax2.text(j, i, f"{Corr_liq[i,j]:.2f}",
                 ha="center", va="center", fontsize=7,
                 color="white" if abs(Corr_liq[i,j]) > 0.35 else "black")


# ── Figure 3: 60-day Mean Bias ──
ax3 = fig.add_subplot(gs[1, 0])
avg_60 = [liquidity_df[n].mean()/1e6 for n in selected]
current = [liquidity_df[n].iloc[-1]/1e6 for n in selected]
overest = [(a/c - 1)*100 for a, c in zip(avg_60, current)]

bar_colors = ["#F44336" if o > 0 else "#4CAF50" for o in overest]
bars = ax3.bar(selected, overest, color=bar_colors, alpha=0.85)
ax3.axhline(0, color="black", lw=0.8)

for bar, val in zip(bars, overest):
    ax3.text(bar.get_x() + bar.get_width()/2,
             val + (2 if val > 0 else -4),
             f"{val:+.1f}%", ha="center", fontsize=8, fontweight="bold")

ax3.set_title("60-Day Mean Bias vs Current TVL\n"
              "Red = Overestimate (too aggressive)\n"
              "Green = Underestimate (too conservative)", fontsize=10)
ax3.set_ylabel("Bias (%)")
ax3.set_xticklabels(selected, rotation=45, fontsize=8)
ax3.grid(True, alpha=0.2, axis="y")


# ── Figure 4: Eigenvalue Spectrum ──
ax4 = fig.add_subplot(gs[1, 1])
ax4.bar(range(1, len(eigenvalues)+1), eigenvalues,
        color=["#F44336" if ev > lambda_max_mp else "#90A4AE"
               for ev in eigenvalues],
        alpha=0.85)

ax4.axhline(lambda_max_mp, color="red", linestyle="--", lw=1.5,
            label=f"MP bound = {lambda_max_mp:.2f}")

ax4.set_xticks(range(1, len(eigenvalues)+1))
ax4.set_xticklabels([f"λ{i}" for i in range(1, len(eigenvalues)+1)])
ax4.set_title(f"Liquidity Matrix Eigenvalue Spectrum\n"
              f"Effective Rank = {eff_rank:.2f}/{N} ({eff_rank/N*100:.0f}% independent)",
              fontsize=10)
ax4.set_ylabel("Eigenvalue")
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.2, axis="y")


# ── Figure 5: Debt Ceiling Recommendation ──
ax5 = fig.add_subplot(gs[1, 2])

debt_ceiling_100 = 100  # Static method baseline: $100M
adjustments = []

for name in selected:
    pct_chg = (liquidity_df[name].iloc[-1] / liquidity_df[name].iloc[0] - 1)
    if pct_chg < -0.2:      # Sharp decline >20% → aggressive haircut
        adj = 0.70
    elif pct_chg < 0:       # Mild decline → moderate haircut
        adj = 0.85
    else:                   # Growing → no haircut (conservative)
        adj = 1.00
    adjustments.append(adj * debt_ceiling_100)

x = range(len(selected))
ax5.bar([i - 0.2 for i in x], [debt_ceiling_100]*len(selected),
        0.38, label="Static Method ($100M baseline)",
        color="steelblue", alpha=0.75)
ax5.bar([i + 0.2 for i in x], adjustments,
        0.38, label="Trend-Adjusted (RMT)",
        color="#F44336", alpha=0.85)

for i, (adj, name) in enumerate(zip(adjustments, selected)):
    if adj < debt_ceiling_100:
        ax5.text(i + 0.2, adj + 1,
                 f"{adj:.0f}M\n({adj-100:+.0f}%)",
                 ha="center", fontsize=7, color="darkred")

ax5.set_xticks(list(x))
ax5.set_xticklabels(selected, rotation=45, fontsize=8)
ax5.set_ylabel("Debt Ceiling ($ Million, normalized)")
ax5.set_title("Debt Ceiling Recommendation\n"
              "Trend-adjusted vs Static 60-day Mean", fontsize=10)
ax5.legend(fontsize=8)
ax5.grid(True, alpha=0.2, axis="y")


# ── Overall Title ──
plt.suptitle(
    "PT Market Liquidity Competition: RMT Analysis of LlamaRisk's Debt Ceiling\n"
    "60-day mean overestimates liquidity in declining markets by up to 61%",
    fontsize=12, y=1.01
)

plt.savefig("pt_liquidity_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

print("Saved: pt_liquidity_analysis.png")